In [ ]:
import sys
sys.stdout = open('/dev/stdout', 'w')

In [ ]:
#!/bin/bash
!curl -L https://www.kaggle.com/api/v1/datasets/download/abdurrahman22224/smartphone-new-data -o data.zip --skip-existing
!unzip -o data.zip
!ls -lah

In [ ]:
import requests

token = requests.post(
    "https://keycloak.k8s-jagr.duckdns.org/realms/platform/protocol/openid-connect/token",
    data={
        "grant_type": "password",
        "client_id": "platform-app",
        "username": "alice",
        "password": "admin",
    },
).json()["access_token"]

In [ ]:
import os
import pyspark
from pyspark.conf import SparkConf
from pyspark.sql import SparkSession

SPARK_VERSION = pyspark.__version__
SPARK_MINOR_VERSION = '.'.join(SPARK_VERSION.split('.')[:2])
ICEBERG_VERSION = "1.7.0"

catalog_name = "sources"
iceberg_warehouse = "sources"

s3_endpoint_host = "https://garage.k8s-jagr.duckdns.org"
s3_region = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
aws_access_key = "GK0123456789abcdef01234567"
aws_secret_key = "0123456789abcdef0123456789abcdef0123456789abcdef0123456789abcdef"

packages = ",".join([
    "software.amazon.awssdk:bundle:2.25.53",
    "software.amazon.awssdk:url-connection-client:2.25.53",
    "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1",
])

config = {
    "spark.sql.defaultCatalog": catalog_name,
    f"spark.sql.catalog.{catalog_name}": "org.apache.iceberg.spark.SparkCatalog",
    f"spark.sql.catalog.{catalog_name}.type": "rest",
    f"spark.sql.catalog.{catalog_name}.uri": "https://lakekeeper.k8s-jagr.duckdns.org/catalog/",
    f"spark.sql.catalog.{catalog_name}.token": token,

    f"spark.sql.catalog.{catalog_name}.warehouse": iceberg_warehouse,
    f"spark.sql.catalog.{catalog_name}.io-impl": "org.apache.iceberg.aws.s3.S3FileIO",
    f"spark.sql.catalog.{catalog_name}.s3.endpoint": s3_endpoint_host,
    f"spark.sql.catalog.{catalog_name}.s3.path-style-access": "true",
    f"spark.sql.catalog.{catalog_name}.s3.region": s3_region,
    f"spark.sql.catalog.{catalog_name}.warehouse": iceberg_warehouse,
    f"spark.sql.catalog.{catalog_name}.s3.access-key-id": aws_access_key,
    f"spark.sql.catalog.{catalog_name}.s3.secret-access-key": aws_secret_key,
    
    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    "spark.jars.packages": packages
}

spark_config = SparkConf().setMaster('local').setAppName("Iceberg-REST")
for k, v in config.items():
    spark_config = spark_config.set(k, v)

spark = SparkSession.builder.config(conf=spark_config).getOrCreate()

spark.sql(f"USE {catalog_name}")


In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("./smartphone_cleaned_v5.csv")
)

print("Rows in source:", df.count())

In [ ]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS smartphone")
table_ident = "smartphone.data"

if not spark.catalog.tableExists(table_ident):
    (
        df.limit(0)
        .writeTo(table_ident)
        .using("iceberg")
        .tableProperty("write.format.default", "parquet")
        .create()
    )
    print("Created table:", table_ident)
else:
    print("Table already exists:", table_ident)